## 1. Project overview

YOLO26n Blood Cell Detection: a reproducible lightweight pipeline. This walkthrough reads frozen evidence; training is opt-in, and final-test execution is not distributed.

In [ ]:
from pathlib import Path
import sys, json
RUN_TRAINING = False
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'docs/evidence/frozen_results.json').exists())
sys.path.insert(0, str(root/'scripts'))
import render_results as results
r = results.load()
print(r['environment'])

## 2. Dataset

Pinned BCCD images and VOC annotations are acquired separately. Raw bytes remain unchanged.

In [ ]:
print(json.dumps(r['dataset'], indent=2))

## 3. Data audit

Two exact zero-area RBC boxes and one same-field acquisition group were found. 66,066 pairs were screened; corrected membership is 206/86/72. Patient/slide independence remains unknown.

In [ ]:
provenance = json.loads((root/'docs/evidence/provenance.json').read_text())
scan = json.loads((root/'docs/evidence/near_duplicate_scan.json').read_text())
print(provenance['upstream_commit'])
print({k:scan[k] for k in ('images','total_pairs_checked','candidates')})

## 4. VOC → YOLO preparation

Continuous numeric endpoints convert to normalized xywh without clipping or ±1 changes. Only registered invalid boxes are filtered from modeling labels.

In [ ]:
from _bootstrap import ROOT
from blood_cell_detection.data import label_text
print(label_text([('RBC', (10,20,110,120))]).decode())
print('Frozen maximum round-trip pixel error:', r['dataset']['round_trip'])

## 5. Experimental setup

Seed 42, batch 16, AMP, maximum 100 epochs, patience 20, independent official pretrained initialization. E1→E2 changes class-weight power; E2→E3 changes input size.

In [ ]:
print(results.blocks(r)['DESIGN'])
print(r['augmentation'])

## 6. E1 / E2 / E3

Fresh reproduction writes to outputs/reproduction and cannot replace published evidence. An engineering smoke check and explicit command are required before training. No test execution is exposed.

In [ ]:
if RUN_TRAINING:
    raise RuntimeError('Use the explicit audited reproduction commands in docs/experiments.md; this walkthrough does not execute models.')
print({i:(e['training']['actual_epochs'], e['training']['best_epoch']) for i,e in r['experiments'].items()})

## 7. Validation comparison

E2 won by a small margin under the preregistered independent validation AP50-95 rule. Class weighting did not consistently improve minority-class AP.

In [ ]:
from IPython.display import display, Markdown
display(Markdown(results.blocks(r)['VALIDATION']))
display(Markdown(results.blocks(r)['CLASSES']))

## 8. Final test

One evaluation after selection freeze and selected-checkpoint restoration. P/R use the max-F1 operating point; AP is curve-integrated. No rerun button exists.

In [ ]:
assert r['test']['model_test_evaluation_calls'] == 1
display(Markdown(results.blocks(r)['FINAL']))
display(Markdown(results.blocks(r)['GAP']))

## 9. Efficiency analysis

Five fresh-process warmed batch-16 FP16 repeats on validation, synchronized CUDA timing. Historical training resources are a separate measurement scope.

In [ ]:
display(Markdown(results.blocks(r)['EFFICIENCY']))

## 10. Error analysis

Counts use confidence ≥0.25 / same-class IoU ≥0.5 greedy matching, a different operating point from aggregate P/R. Existing annotation coverage may be incomplete.

In [ ]:
display(Markdown(results.blocks(r)['ERRORS']))
from IPython.display import Image
for name in ('prediction_good.png','prediction_high_fn.png','prediction_high_fp.png'):
    display(Image(filename=str(root/'docs/assets'/name), embed=False))

## 11. Limitations

Small dataset, one seed, one split, missing patient/slide metadata and limited test/minority samples. No clinical validation; benchmarks are hardware dependent.

In [ ]:
display(Markdown((root/'docs/limitations.md').read_text()))
print('Model execution in default walkthrough: 0')